In [ ]:
!pip install mlflow -q
!pip install --upgrade transformers -q
!pip uninstall -y torchao -q


In [ ]:
import json, random, torch, numpy as np
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score, recall_score
import mlflow
from datasets import Dataset, load_dataset, concatenate_datasets, Value
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, pipeline, TrainerCallback, 
    EarlyStoppingCallback)
from transformers import logging as hf_logging
from huggingface_hub import login, HfApi, create_repo, hf_hub_download, repo_exists
from peft import LoraConfig, get_peft_model, TaskType
import os
import glob
hf_logging.set_verbosity_error()

# setup of kaggle secrets in notebook programmatically mutuated 
# from this guide https://www.kaggle.com/discussions/product-feedback/666571
# secrets dataset automatically updated in in automated retraining workflow step
def get_secret(label, filename="secrets.json"):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        pass

    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches:
        with open(matches[0]) as f:
            return json.load(f)[label]

    listing = os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "MISSING"
    raise ValueError(f"Secret '{label}' not found. /kaggle/input contains: {listing}")

HF_USER =           get_secret("HF_USER")
HF_TOKEN =          get_secret("HF_TOKEN")
SPACE_URL =         get_secret("SPACE_URL")
SYNTH_DATA_REPO =   HF_USER + "/" + get_secret("SYNTH_DATA_REPO")
MODEL_REPO   =      HF_USER + "/" + get_secret("MODEL_REPO")
METRICS_REPO =      HF_USER + "/" + get_secret("METRICS_REPO")
SPACE_NAME =        get_secret("SPACE_NAME")

login(token=HF_TOKEN)

COMPANY       = get_secret("COMPANY")
COMPANY_DESC  = get_secret("COMPANY_DESC")
BASE_MODEL    = "cardiffnlp/twitter-roberta-base-sentiment-latest"
N_PER_CLASS   = 100
LABEL_MAP     = {"negative": 0, "neutral": 1, "positive": 2}

# Aspect Based Sentiment Analysis (ABSA) -> will relate the specific aspect with its sentiment in the text
ASPECT = COMPANY 


MODEL_EXISTS = HF_USER and MODEL_REPO and repo_exists(MODEL_REPO)

# set tracking URI to rerouted MLFLOW URI in huggingface space
mlflow.set_tracking_uri(f"{SPACE_URL}/mlflow")
create_repo(METRICS_REPO, repo_type="dataset", exist_ok=True)


RepoUrl('https://huggingface.co/datasets/divde/sentiment-training-metrics', endpoint='https://huggingface.co', repo_type='dataset', repo_id='divde/sentiment-training-metrics')

In [ ]:
# absa specific dataset split only from train as test split is an empty dataset, planned to test
# for term extraction (another absa nlp task) and labelling in competitions
absa_restaurants = load_dataset("tomaarsen/setfit-absa-semeval-restaurants",split="train")
absa_laptops = load_dataset("tomaarsen/setfit-absa-semeval-laptops",split="train")

# align datasets schema for caoncatenation with other data
def clean_absa(dataset):
    dataset = dataset.filter(lambda x: x["label"] in LABEL_MAP)
    dataset = dataset.remove_columns(["ordinal"])
    return dataset.map(lambda x: {"label": LABEL_MAP[x["label"]]})

absa_data = concatenate_datasets([clean_absa(absa_restaurants), clean_absa(absa_laptops)])

# multiple split for eval and final test
split1 = absa_data.train_test_split(test_size=0.3, seed=42)
train_data = split1["train"]
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)
val_data = split2["train"]
real_test = split2["test"]


MAX_TEST = 1000
if len(real_test) > MAX_TEST:
    real_test = real_test.train_test_split(train_size=MAX_TEST, seed=42)["train"]


In [ ]:
# fallback to base model for first training
if MODEL_EXISTS:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, add_prefix_space=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

# tokenization for domain specific aspect for generated data
def aspect_tokenize(aspect):
    def _tokenize(batch):
        return tokenizer(batch["text"],[aspect] * len(batch["text"]), max_length=128, truncation=True)
    return _tokenize
# tokenization for dynamic aspect from absa dataset
def tokenize_per_example(batch):
    return tokenizer(batch["text"], batch["span"], max_length=128, truncation=True)


In [ ]:
# loading of synthetic dataset datapoint from personal HF repo created from generation notebook in the project
# dataset split in generated "synthetic" and automatically labelled "weak labelled" by Gemma instance 
# with controlled fallback to empty dataset
try:
    synth_pool = load_dataset(SYNTH_DATA_REPO)["train"]
    gen_dataset = (synth_pool.filter(lambda x: x["source"] == "synthetic")
                   .remove_columns(["source"])
                   .map(aspect_tokenize(ASPECT), batched=True))
    posts_labeled_dataset = (synth_pool.filter(lambda x: x["source"] == "weak_labelled")
                              .remove_columns(["source"])
                              .map(aspect_tokenize(ASPECT), batched=True))
except Exception:
    print("No synthetic data yet, run data_generation.ipynb first")
    gen_dataset = Dataset.from_list([])
    posts_labeled_dataset = Dataset.from_list([])


In [ ]:
# Balancing of synthetic and real data through ratio and sampling
SYNTH_RATIO = 0.2  
n_synth = len(gen_dataset)
n_real = int(n_synth * (1 - SYNTH_RATIO) / SYNTH_RATIO) if n_synth > 0 else 1500
n_per_label = n_real // 3

sampled_indices = []
for label in [0, 1, 2]:
    label_indices = [i for i, l in enumerate(train_data["label"]) if l == label]
    sampled_indices.extend(random.choices(label_indices, k=n_per_label))

# column casting to the same datatype of the other dataset and cleanup
train_data_sampled = train_data.select(sampled_indices).cast_column("label", Value("int64"))
train_data_sampled = train_data_sampled.map(tokenize_per_example, batched=True)
train_data_sampled = train_data_sampled.remove_columns(["span"])

val_data = val_data.map(tokenize_per_example, batched=True)
real_test = real_test.map(tokenize_per_example, batched=True)

datasets_to_combine = [gen_dataset, train_data_sampled]
if len(gen_dataset) > 0:
    datasets_to_combine.append(gen_dataset)
if len(posts_labeled_dataset) > 0:
    datasets_to_combine.append(posts_labeled_dataset)

combined_train = concatenate_datasets(datasets_to_combine)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Train: {len(combined_train)} | Val: {len(val_data)} | Real test: {len(real_test)}")


In [ ]:
# log each evaluation against validation step 
class MLflowCallback(TrainerCallback):
    def on_evaluate(self,args, state,control, metrics= None, **kwargs):
        if metrics:
            mlflow.log_metrics(
                {k: v for k, v in metrics.items() if isinstance(v, (int, float))},
                step=int(state.epoch)
            )


In [ ]:
# add required evaluation metrics 
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"f1": f1_score(labels, preds, average="weighted"),
            "accuracy": accuracy_score(labels,preds),
            "recall": recall_score(labels,preds, average="weighted")}

if MODEL_EXISTS:
    current = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO)
else:
    current = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
    
baseline_metrics = Trainer(
    model=current,
    eval_dataset=real_test,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=collator,
).evaluate()
baseline_f1 = baseline_metrics["eval_f1"]
baseline_accuracy = baseline_metrics["eval_accuracy"]
baseline_recall = baseline_metrics["eval_recall"]
print(f"Baseline F1 : {baseline_f1:.4f} | Baseline Accuracy : {baseline_accuracy:.4f} | Baseline Recall : {baseline_recall:.4f}")


del current
torch.cuda.empty_cache()


In [ ]:
if MODEL_EXISTS:
    new_model = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO, num_labels=3)
else:
    new_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=3)

# PEFT - LoRA training -> insert new smaller weight matrices that becomes the trainable parameters
# allowing faster finetuning of the model 
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, #rank dimension of the matrices to train
    lora_alpha = 32, # scaling of the rank influence alpha / r
    lora_dropout = 0.1,
    # as per original paper target on all attention layers -good results without increasing computational cost
    target_modules = ["query", "value"], 
)

new_model = get_peft_model(new_model, lora_config)

trainer = Trainer(
    model=new_model,
    args=TrainingArguments(
        output_dir="./results",
        hub_model_id=MODEL_REPO,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-4,
        per_device_train_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        load_best_model_at_end=True,
    ),
    train_dataset=combined_train,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=collator,
    callbacks=[MLflowCallback()],
)



In [ ]:
run_timestamp = datetime.now().isoformat()

with mlflow.start_run(run_name = f"v{run_timestamp}"):

    trainer.train()
    new_metrics = trainer.evaluate(eval_dataset= real_test)
    
    new_f1 = new_metrics["eval_f1"]
    new_accuracy = new_metrics["eval_accuracy"]
    new_recall = new_metrics["eval_recall"]
    print(f"New F1: {new_f1:.4f} | New Accuracy : {new_accuracy:.4f} | New Recall : {new_recall:.4f}")
        
    quality_control = new_f1 > baseline_f1

    print(f"Ready for service: {quality_control}  ({baseline_f1:.4f} → {new_f1:.4f})")

    api = HfApi(token=HF_TOKEN)
    
    # JSON with history of the training run
    payload = {
        "timestamp":            datetime.now().isoformat(),
        "baseline_f1":          round(baseline_f1, 4),
        "baseline_accuracy":    round(baseline_accuracy,4),
        "baseline_recall":      round(baseline_recall,4),
        "new_f1":               round(new_f1, 4),
        "new_accuracy":         round(new_accuracy,4),
        "new_recall":           round(new_recall,4),
        "performance":          quality_control,
        "n_samples":            len(combined_train),
    }
    try:
        path = hf_hub_download(repo_id=METRICS_REPO, filename="history.json", repo_type="dataset")
        history = json.load(open(path))
    except:
        history = []

    history.append(payload)

    # Upload file to HF Space dataset
    api.upload_file(
    path_or_fileobj=json.dumps(history).encode(),
    path_in_repo="history.json",
    repo_id=METRICS_REPO,
    repo_type="dataset",
    commit_message=f"run {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    )
    # Mlflow logging for model evaluation
    mlflow.log_metric("baseline_f1", baseline_f1)
    mlflow.log_metric("new_f1", new_f1)
    mlflow.log_metric("new_accuracy", new_accuracy)
    mlflow.log_metric("new_recall", new_recall)
    # If model uploaded to HF
    mlflow.set_tag("deployed_to_production", str(quality_control))

    if quality_control:
        # Fix After a Bad request Error from hugginface_hub
        tag_name = f"v{run_timestamp.replace(':', '-')}"

        # Merging of LoRA weight Matrix into the original model before push
        trainer.model = trainer.model.merge_and_unload()

        #huggingface_hub functions for repo and models management and versioning
        commit_info = trainer.push_to_hub(
            commit_message=f"retrain: F1 {baseline_f1:.3f}→{new_f1:.3f}"
        )

        api.create_tag(
            repo_id=MODEL_REPO, 
            tag=tag_name, 
            tag_message=f"F1={new_f1:.3f}",
            repo_type="model"
            )
        mlflow.set_tag("hf_hub_commit", commit_info.oid)
        mlflow.set_tag("hf_hub_tag",tag_name)
        api.restart_space(repo_id=f"{HF_USER}/{SPACE_NAME}")
